# Learning the effect and analysis of corshrink

In [1]:
# התקנות רק אם חסר:
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("ashr", quietly = TRUE)) devtools::install_github("stephenslab/ashr")
if (!requireNamespace("CorShrink", quietly = TRUE)) devtools::install_github("kkdey/CorShrink")

library(CorShrink)


Using GitHub PAT from the git credential store.




magrittr  (2.0.3 -> 2.0.4    ) [CRAN]
gmp       (NA    -> 0.7-5    ) [CRAN]
stringr   (1.5.1 -> 1.5.2    ) [CRAN]
RcppEigen (NA    -> 0.3.4.0.2) [CRAN]
osqp      (NA    -> 0.6.3.3  ) [CRAN]
scs       (NA    -> 3.2.7    ) [CRAN]
ECOSolveR (NA    -> 0.5.5    ) [CRAN]
Rmpfr     (NA    -> 1.1-1    ) [CRAN]
shape     (NA    -> 1.4.6.1  ) [CRAN]
corpcor   (NA    -> 1.6.10   ) [CRAN]
corrplot  (NA    -> 0.95     ) [CRAN]
CVXR      (NA    -> 1.0-15   ) [CRAN]
glmnet    (NA    -> 4.1-10   ) [CRAN]


Installing 13 packages: magrittr, gmp, stringr, RcppEigen, osqp, scs, ECOSolveR, Rmpfr, shape, corpcor, corrplot, CVXR, glmnet

Installing packages into '/Users/edeneldar/Library/R/arm64/4.5/library'
(as 'lib' is unspecified)




The downloaded binary packages are in
	/var/folders/5g/7gzv8rg14tv7prkqk0v2f3280000gn/T//RtmpJqJYaM/downloaded_packages
-- R CMD build -----------------------------------------------------------------
* checking for file '/private/var/folders/5g/7gzv8rg14tv7prkqk0v2f3280000gn/T/RtmpJqJYaM/remotes3f0d173cd2ea/kkdey-CorShrink-a9f6ba0/DESCRIPTION' ... OK
* preparing 'CorShrink':
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building 'CorShrink_0.1-6.tar.gz'



Installing package into '/Users/edeneldar/Library/R/arm64/4.5/library'
(as 'lib' is unspecified)



In [2]:
set.seed(1)

donors_all <- paste0("D", 1:12)
genes      <- paste0("g", 1:5)

A_donors <- paste0("D", 1:10)          # רקמה A: 10 תורמים
B_donors <- c(paste0("D", 1:6), "D9","D10","D11","D12")  # רקמה B: 10 תורמים, חיתוך חלקי

# פונקציה קטנה לבניית וקטורי ביטוי פר-גן עם קורלציה אמיתית ~0.6 על חיתוך התורמים
make_pair <- function(g){
  # אות משותף לתורמים בחיתוך
  overlap <- intersect(A_donors, B_donors)
  z_overlap <- rnorm(length(overlap))
  # A:
  xA <- rnorm(length(A_donors))
  names(xA) <- A_donors
  xA[overlap] <- scale(z_overlap + rnorm(length(overlap), sd=0.8))[,1]
  # B:
  xB <- rnorm(length(B_donors))
  names(xB) <- B_donors
  xB[overlap] <- scale(0.6*z_overlap + rnorm(length(overlap), sd=0.8))[,1]
  list(A=xA, B=xB)
}

pairs_list <- lapply(genes, make_pair)
names(pairs_list) <- genes

# בונים מטריצה Donors × Variables, עם עמודות "A:g1"... "B:g5" ומילוי NA כשחסר
X <- matrix(NA_real_, nrow=length(donors_all), ncol=2*length(genes),
            dimnames=list(donors_all, c(paste0("A:",genes), paste0("B:",genes))))
for(g in genes){
  X[names(pairs_list[[g]]$A), paste0("A:", g)] <- pairs_list[[g]]$A
  X[names(pairs_list[[g]]$B), paste0("B:", g)] <- pairs_list[[g]]$B
}


In [12]:
pairs_list

$g1
$g1$A
         D1          D2          D3          D4          D5          D6 
-0.21158856  0.26752711 -0.31147011  1.45892723  0.06191851 -2.07429185 
         D7          D8          D9         D10 
 1.12493092 -0.04493361  0.51501984  0.29395783 

$g1$B
         D1          D2          D3          D4          D5          D6 
-1.26248474 -0.30545942  0.09549014  1.60465006 -0.30120696 -1.26713778 
         D9         D10         D11         D12 
 0.69377096  0.74237775 -1.37705956 -0.41499456 


$g2
$g2$A
        D1         D2         D3         D4         D5         D6         D7 
-0.3414149 -0.8232370 -0.4232283  0.5873077 -1.5890653  1.5841145  0.5697196 
        D8         D9        D10 
-0.1350546  0.2366993  0.7688240 

$g2$B
         D1          D2          D3          D4          D5          D6 
-1.10028007 -0.71316515  1.24100292 -0.97285086  0.37174146  0.81853304 
         D9         D10         D11         D12 
 1.15808186 -0.80306321  0.07434132 -0.58952095 


$g3
$g3$A
          D1           D2           D3           D4           D5           D6 
-0.052741655 -0.005516226 -1.126111110  1.413768500 -0.318290026 -0.610563650 
          D7           D8           D9          D10 
-0.910921649  0.158028772  1.576838673 -0.877384506 

$g3$B
        D1         D2         D3         D4         D5         D6         D9 
-0.3243554  0.9238677 -1.4404220  0.8479840 -0.3051517 -0.2843694  1.4961641 
       D10        D11        D12 
-0.9137172 -0.2145794 -0.1795565 


$g4
$g4$A
         D1          D2          D3          D4          D5          D6 
 1.54496774 -0.71053190  0.76407047 -1.40506966 -0.67118898 -0.57583913 
         D7          D8          D9         D10 
 2.08716655  0.01739562  0.85320180  0.20038965 

$g4$B
         D1          D2          D3          D4          D5          D6 
 0.25457731 -0.76758597  2.22364338 -0.86698262  0.23478397 -0.37277299 
         D9         D10         D11         D12 
-0.67430472 -0.03135837 -0.25502703 -1.42449465 


$g5
$g5$A
         D1          D2          D3          D4          D5          D6 
-0.69954459  0.80301667  1.80437576 -0.09025902 -1.46682531  0.21096167 
         D7          D8          D9         D10 
-0.17710396  0.40201178  0.07129795 -0.63302313 

$g5$B
         D1          D2          D3          D4          D5          D6 
-0.09071960 -0.09396723  1.10539584  1.61445925 -1.17607777 -0.62919751 
         D9         D10         D11         D12 
 0.37974848 -1.10964145 -0.16437583  0.42069464

In [5]:
# 1) מטריצת TRUE/FALSE למי לא NA
ok <- !is.na(X)
storage.mode(ok) <- "integer"         # כדי שהמכפלה תחזיר ספירות

# 2) n_mat: לכל זוג עמודות — כמה תצפיות משותפות יש
n_mat <- crossprod(ok)                 # שקול ל- t(ok) %*% ok
dimnames(n_mat) <- list(colnames(X), colnames(X))

# 3) בחירה של בלוק A×B בלי להפיל ממדים
ixA <- grepl("^A:", colnames(X))
ixB <- grepl("^B:", colnames(X))
n_cross     <- n_mat[ixA, ixB, drop = FALSE]

# (מומלץ גם ל-naive לשמור ממדים)
naive_cross <- cor_naive[ixA, ixB, drop = FALSE]

# בדיקה על g1:
c(
  naive = naive_cross["A:g1","B:g1"],
  n     = n_cross["A:g1","B:g1"]
)


naive         n 
0.8117187 8.0000000

In [7]:
# שלב 3 — CorShrink על המטריצה X
# (עובד ישירות מהדאטה עם NA ומחשב את ה-n לבד)
cs <- CorShrink::CorShrinkData(X)

# לפעמים הפלט הוא רשימה, נטפל בשני המקרים:
cor_shrunk <- if (is.list(cs)) {
  if (!is.null(cs$cor)) cs$cor else if (!is.null(cs$ash_cor)) cs$ash_cor else stop("Unexpected CorShrinkData output")
} else {
  as.matrix(cs)
}

# נחזיר בלוק A×B בלי להפיל ממדים:
ixA <- grepl("^A:", colnames(X))
ixB <- grepl("^B:", colnames(X))
shrunk_cross <- cor_shrunk[ixA, ixB, drop = FALSE]

# השוואה נקודתית לאותו זוג-גן:
c(
  naive  = naive_cross["A:g1","B:g1"],
  shrunk = shrunk_cross["A:g1","B:g1"],
  n      = n_cross["A:g1","B:g1"]
)


naive    shrunk         n 
0.8117187 0.2079011 8.0000000

In [13]:
r <- naive_cross["A:g1","B:g1"]
n <- n_cross["A:g1","B:g1"]
z  <- atanh(r)
se <- 1/sqrt(n - 3)
ci_z  <- c(z - 1.96*se, z + 1.96*se)
ci_r  <- tanh(ci_z)
round(c(r_naive=r, n=n, ci_lo=ci_r[1], ci_hi=ci_r[2]), 3)


r_naive       n   ci_lo   ci_hi 
  0.812   8.000   0.250   0.965

In [14]:
genes <- paste0("g", 1:5)

summ <- t(sapply(genes, function(g){
  r  <- naive_cross[paste0("A:",g), paste0("B:",g)]
  n  <- n_cross[paste0("A:",g), paste0("B:",g)]
  z  <- atanh(r); se <- 1/sqrt(max(n - 3, 1))
  ci <- tanh(c(z - 1.96*se, z + 1.96*se))
  shr <- shrunk_cross[paste0("A:",g), paste0("B:",g)]
  c(r_naive=r, n=n, ci_lo=ci[1], ci_hi=ci[2], r_shrunk=shr, delta=shr - r)
}))

round(summ, 3)


,r_naive,n,ci_lo,ci_hi,r_shrunk,delta
g1,0.812,8,0.250,0.965,0.208,-0.604
g2,0.033,8,-0.688,0.721,0.000,-0.032
g3,0.895,8,0.515,0.981,0.654,-0.241
g4,0.476,8,-0.344,0.884,0.012,-0.464
g5,0.594,8,-0.190,0.916,0.025,-0.570


In [15]:
# ניצור עותק ונשמיט בכוונה ערכים בתוך החיתוך:
X2 <- X
overlap <- intersect(A_donors, B_donors)

set.seed(42)
# נוריד 4 תורמים חופפים מהעמודה A:g2
dropA_g2 <- sample(overlap, 4, replace = FALSE)
X2[dropA_g2, "A:g2"] <- NA

# נוריד 5 תורמים חופפים מהעמודה B:g3
dropB_g3 <- sample(overlap, 5, replace = FALSE)
X2[dropB_g3, "B:g3"] <- NA

# קורלציות נאיביות + מטריצת n מעודכנות
cor_naive2 <- cor(X2, use = "pairwise.complete.obs")
ok2 <- !is.na(X2); storage.mode(ok2) <- "integer"
n_mat2 <- crossprod(ok2); dimnames(n_mat2) <- list(colnames(X2), colnames(X2))

ixA <- grepl("^A:", colnames(X2))
ixB <- grepl("^B:", colnames(X2))
naive_cross2 <- cor_naive2[ixA, ixB, drop = FALSE]
n_cross2     <- n_mat2[ixA, ixB, drop = FALSE]

# CorShrink על הדאטה עם החוסרים החדשים
cs2 <- CorShrink::CorShrinkData(X2)
cor_shrunk2 <- if (is.list(cs2)) {
  if (!is.null(cs2$cor)) cs2$cor else if (!is.null(cs2$ash_cor)) cs2$ash_cor else stop("Unexpected CorShrinkData output")
} else as.matrix(cs2)
shrunk_cross2 <- cor_shrunk2[ixA, ixB, drop = FALSE]

# מסכמים לשלושה גנים (g1 בסיס, g2/g3 עם n קטן יותר)
genes <- c("g1","g2","g3")
summ2 <- t(sapply(genes, function(g){
  r   <- naive_cross2[paste0("A:",g), paste0("B:",g)]
  n   <- n_cross2[paste0("A:",g),    paste0("B:",g)]
  shr <- shrunk_cross2[paste0("A:",g), paste0("B:",g)]
  c(r_naive = r, n = n, r_shrunk = shr, delta = shr - r)
}))
round(summ2, 3)


,r_naive,n,r_shrunk,delta
g1,0.812,8,0.136,-0.676
g2,-0.071,4,-0.001,0.070
g3,0.995,3,0.156,-0.839


In [16]:
genes <- c("g1","g2","g3")
check <- t(sapply(genes, function(g){
  r   <- naive_cross2[paste0("A:",g), paste0("B:",g)]
  n   <- n_cross2[paste0("A:",g),    paste0("B:",g)]
  se  <- if (n > 3) 1/sqrt(n - 3) else Inf
  z   <- if (is.finite(se)) abs(atanh(r)/se) else NA_real_  # עוצמת ראיה
  shr <- shrunk_cross2[paste0("A:",g), paste0("B:",g)]
  c(n=n, r_naive=r, r_shrunk=shr, delta=shr-r, z_abs=z)
}))
round(check, 3)


,n,r_naive,r_shrunk,delta,z_abs
g1,8,0.812,0.136,-0.676,2.531
g2,4,-0.071,-0.001,0.070,0.071
g3,3,0.995,0.156,-0.839,NA


In [17]:
# נעלה את החיתוך (n) של g1 מ-8 ל-10 ע"י מילוי ערכים חסרים ל-A:g1 בתורמים D11,D12
X3 <- X2

A_donors <- paste0("D",1:10)
B_donors <- c(paste0("D",1:6),"D9","D10","D11","D12")
overlap2 <- intersect(A_donors, B_donors)          # D1-6, D9, D10
new_A_donors <- setdiff(B_donors, A_donors)        # D11, D12

# נמלא את A:g1 ל-D11,D12 לפי הממוצע/סטיית התקן של A:g1 בחיתוך הקיים
mu  <- mean(X2[overlap2, "A:g1"], na.rm=TRUE)
sig <- sd( X2[overlap2, "A:g1"], na.rm=TRUE)
set.seed(123)
X3[new_A_donors, "A:g1"] <- rnorm(length(new_A_donors), mu, sig)

# מחשבים מחדש קורלציות נאיביות ו-n
cor_naive3 <- cor(X3, use="pairwise.complete.obs")
ok3 <- !is.na(X3); storage.mode(ok3) <- "integer"
n_mat3 <- crossprod(ok3); dimnames(n_mat3) <- list(colnames(X3), colnames(X3))

ixA <- grepl("^A:", colnames(X3))
ixB <- grepl("^B:", colnames(X3))
naive_cross3 <- cor_naive3[ixA, ixB, drop=FALSE]
n_cross3     <- n_mat3[ixA, ixB, drop=FALSE]

# CorShrink מחדש
cs3 <- CorShrink::CorShrinkData(X3)
cor_shrunk3 <- if (is.list(cs3)) {
  if (!is.null(cs3$cor)) cs3$cor else if (!is.null(cs3$ash_cor)) cs3$ash_cor else stop("Unexpected CorShrinkData output")
} else as.matrix(cs3)
shrunk_cross3 <- cor_shrunk3[ixA, ixB, drop=FALSE]

# השוואה של g1 לפני/אחרי
res <- rbind(
  before = c(n=n_cross2["A:g1","B:g1"], naive=naive_cross2["A:g1","B:g1"], shrunk=shrunk_cross2["A:g1","B:g1"], delta=shrunk_cross2["A:g1","B:g1"]-naive_cross2["A:g1","B:g1"]),
  after  = c(n=n_cross3["A:g1","B:g1"], naive=naive_cross3["A:g1","B:g1"], shrunk=shrunk_cross3["A:g1","B:g1"], delta=shrunk_cross3["A:g1","B:g1"]-naive_cross3["A:g1","B:g1"])
)
round(res, 3)


,n,naive,shrunk,delta
before,8,0.812,0.136,-0.676
after,10,0.802,0.421,-0.381


In [18]:
library(CorShrink)

# XA: donors×genes for tissue A
# XB: donors×genes for tissue B
# returns: list with shrunk cross-correlation and adjacency (= |r|^beta)
cross_block_from_CorShrink <- function(XA, XB, beta = 2) {
  stopifnot(!is.null(rownames(XA)), !is.null(rownames(XB)))
  G <- intersect(colnames(XA), colnames(XB))
  if (length(G) < 2) stop("Need >=2 shared genes across tissues")
  D <- union(rownames(XA), rownames(XB))

  X <- matrix(NA_real_, nrow = length(D), ncol = 2*length(G),
              dimnames = list(D, c(paste0("A:", G), paste0("B:", G))))

  # fill per gene to keep alignment simple & robust
  for (g in G) {
    if (g %in% colnames(XA)) X[rownames(XA), paste0("A:", g)] <- XA[, g]
    if (g %in% colnames(XB)) X[rownames(XB), paste0("B:", g)] <- XB[, g]
  }

  cs <- CorShrinkData(X)
  cor_shrunk <- if (is.list(cs)) (cs$cor %||% cs$ash_cor) else as.matrix(cs)

  ixA <- grepl("^A:", colnames(cor_shrunk))
  ixB <- grepl("^B:", colnames(cor_shrunk))
  R_ab <- cor_shrunk[ixA, ixB, drop = FALSE]            # |genes_A| × |genes_B|
  A_ab <- abs(R_ab)^beta

  list(R_cross = R_ab, A_cross = A_ab)
}

`%||%` <- function(a,b) if (!is.null(a)) a else b
